# EW4 data - GPM IMERG - Training an Autoencoder
This notebook explore the GPM IMERG dataset, including loading in PyEarthTools.

This dataset is a subset of the [NASA Integrated Multi-satellite Retrieval (IMERG)](https://gpm.nasa.gov/data/imerg) data for Global Precipitation Measurement (GPM). This subset has the following extents:
- Temporal Extent: April to September 2025
- Spatial Extent
  - Latitude 0N to 20N
  - Longitude 27W to 20E


### Import libraries

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pathlib
import datetime
import functools
import math

In [3]:
import numpy

In [4]:
import xarray

In [5]:
import matplotlib
import cartopy.crs

In [6]:
import site_archive_jasmin

ROOT_DIRECTORIES: {'ERA5lowres': '/gws/nopw/j04/mohc_shared/dscop/weatherbench/5.625deg/', 'MOGLOBAL': '/gws/nopw/j04/mohc_shared//dscop/mo_pet_site_archive/mo_global/', 'MOUKV': '/gws/nopw/j04/mohc_shared//dscop/mo_pet_site_archive/mo_ukv/', 'Himawari': '/gws/ssde/j25a/mmh_storage/theme3/rv74_himawari', 'HimawariChannels': '/gws/ssde/j25a/mmh_storage/theme3/ra22_himawari', 'Rainfields3': '/gws/ssde/j25a/mmh_storage/theme3/rq0Radar', 'ew4_imerg_precip': '/gws/nopw/j04/ew4energy/imerg_2025_summer', 'ew4_merra2_meteo': '/gws/nopw/j04/ew4energy/West_Africa_Merra-2_2015-2025_meteo', 'ew4_merra2_aero': '/gws/nopw/j04/ew4energy/West_Africa_Merra-2_2015-2025_aerosols/3d', 'ew4_mtg_li': '/gws/nopw/j04/ew4energy/MTG_LI/', 'ew4_era5': '/gws/nopw/j04/ew4energy/ERA5/tutorial_202606'}


In [7]:
import pyearthtools

In [8]:
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe

In [9]:
from pyearthtools.data import Petdt, TimeDelta
from pyearthtools.data.exceptions import DataNotFoundError
from pyearthtools.data.indexes import ArchiveIndex, decorators
from pyearthtools.data.transforms import Transform, TransformCollection
from pyearthtools.data.archive import register_archive


In [10]:
from site_archive_jasmin.utilities import (
    cached_exists,
    cached_iterdir,
)  # Could these be moved into a generic module?


In [11]:
import torch

## Explore the data in the dataset
Lets start by looking at a sample data file, before demonstrating loading the data in pyearthtools. From this we can learn the following things about the dataset
* The pattern of filenames for accessing the data
* The spatial and tempoeral extents of the data
* The variables present in the data
* Any potential problems with its usage, which can be fixed in the PyEarthTools data accessor class
* Plot some data so we can check that PyEarthTools loads it correctly.

We'll be using the IMERG data in the EW4 Group Workspace. It is also available through other sources:
* [NASA](https://gpm.nasa.gov/data/imerg)
* [AWS Open Data](https://registry.opendata.aws/nasa-gpm3imergm/)
* [CEDA Archive - available on JASMIN](https://catalogue.ceda.ac.uk/uuid/47c32530265d4d6e8fdb6c08b2330371/)


## Loading the Data in PyEarthTools
Now we will load the data through the PyEarthTools data accessor. This is a class that has been created which how to access the data. For a dataset like this which is stored as files on disk, it specifries which files to load. Data could be loaded through other mechanisms though, for example it could be load from a database, through a web api, from a tape archive,  from a cloud-based object store or many other mechanisms.

We will take a look at the construction of the accessor, which can be found here:
* [IMERG Data Accessor](https://github.com/MetOffice/pyearthtools_jasmin/blob/main/src/site_archive_jasmin/ew4_imerg.py)
* [Template for Data Accessors](https://github.com/MetOffice/pyearthtools_jasmin/blob/main/src/site_archive_jasmin/template_accessor.py)

Further Reading 
* [
* [PyEarthTools and Data Access](https://pyearthtools.readthedocs.io/en/latest/data.html)
* [Data Module Deep Dive Tutorials](https://pyearthtools.readthedocs.io/en/latest/notebooks/Gallery.html#Deep-Dive---The-Data-Module)


In [12]:
ew4_imerg_accessor = site_archive_jasmin.Ew4Imerg('2025-04-01 00:00', '2025-10-01 00:00')

In [13]:
ghana_extents = {
    'latitude': (4.7,11.1),
    'longitude': (-3.5, 2.9),
}


In [14]:
ghana_pet_box = (ghana_extents['latitude'][0],
                 ghana_extents['latitude'][1],
                 ghana_extents['longitude'][0],
                 ghana_extents['longitude'][1],
                )

## Train an ML model
To demonstrate how we can use this data in a machine learning training pipeline, we will show a simple autoencoder

In [15]:
ew4_imerg_prep = petpipe.Pipeline(
    ew4_imerg_accessor,
    petdata.transform.region.Bounding(*ghana_pet_box),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

In [16]:
ew4_imerg_ml = petpipe.Pipeline(
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
)


In [17]:
(ew4_imerg_prep | ew4_imerg_ml)['2025-07-05 16:00']

/home/users/shaddad/prog/pet_fork/packages/data/src/pyearthtools/data/indexes/_indexes.py:779: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


array([[[[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]]]],
      shape=(1, 1, 64, 64), dtype=float32)

In [36]:
train_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250501T00', '20250701T00', interval='1 hour').randomise(), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)
val_range = petpipe.Pipeline(
    petpipe.modifications.TemporalWindow(prior_indexes=[0,], posterior_indexes=[0,], timedelta=TimeDelta('30 minutes')),
    iterator=petpipe.iterators.DateRange('20250701T00', '20250801T00', interval='1 hour'), 
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

Calculated indexes


In [37]:
ew4_imerg_train_pipe = ew4_imerg_prep | ew4_imerg_ml | train_range
ew4_imerg_val_pipe = ew4_imerg_prep | ew4_imerg_ml | val_range

In [38]:
numpy.histogram(ew4_imerg_val_pipe['2025-07-28 18:00'])


/home/users/shaddad/prog/pet_fork/packages/data/src/pyearthtools/data/indexes/_indexes.py:779: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


(array([8148,   30,    6,    0,    4,    0,    0,    2,    0,    2]),
 array([0.       , 0.975    , 1.95     , 2.9250002, 3.9      , 4.875    ,
        5.8500004, 6.8250003, 7.8      , 8.775001 , 9.75     ],
       dtype=float32))

## Set up autoencoder architecture

In [39]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

### Determine the size of the filters around the latent space
One data dpeendant thing is to determine the size of the convolutional blocks either side of the latent space. We create some dummy layer to apply to our data to determine the size.

In [40]:
sample_tensor = torch.tensor(next(iter(ew4_imerg_train_pipe))[0][0], dtype=torch.float32).to(device)

In [41]:
sample_tensor

tensor([[[[0.0000, 0.0300, 0.0100,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0100, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]]],
       device='cuda:0')

In [42]:
sample_tensor.shape

torch.Size([1, 1, 64, 64])

In [43]:
torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=1, 
                            out_channels=16, 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=16, 
                            out_channels=32, 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
).to(device)(sample_tensor).shape

torch.Size([1, 32, 16, 16])

### Create model class

In [44]:
class ImergAutoEncoder(torch.nn.Module):
    def __init__(self, input_channels, max_pool=False):
        super(ImergAutoEncoder, self).__init__()

        # we have "hard coded" a lot of the architecture hyperparameters in our model class. 
        # Usually you want want to make these arguments for the class so you can vary hyperparameters more easily.
        # Hard coding here makes it easier to follow the architecture definition in the tutorial
        
        self._num_channels = [input_channels, 16,32]
        self._latent_array_dims = (-1,self._num_channels[-1],16,16)
        self._prelatent_size = functools.reduce(lambda a,b:a*b, self._latent_array_dims[1:])
        # self._latent_size = 500
        
        self._encoder = self._get_encoder(max_pool)
        self._decoder = self._get_decoder()

    def _get_encoder(self, max_pool):

        encoder = torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=self._num_channels[0], 
                            out_channels=self._num_channels[1], 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Conv2d(in_channels=self._num_channels[1], 
                            out_channels=self._num_channels[2], 
                            kernel_size=3, 
                            padding=1,
                            stride=2,
                           ),
            torch.nn.ReLU(),
            torch.nn.Flatten(1, -1),
            )
        return encoder

    def _get_decoder(self):
        """
        """
        decoder = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(in_channels=self._num_channels[2], out_channels=self._num_channels[1], kernel_size=2,stride=2),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(in_channels=self._num_channels[1], out_channels=self._num_channels[0], kernel_size=2,stride=2),
        )
        return decoder
        
    def forward(self, x):

        # Get latent representation
        latent = self._encoder(x)

        # Reconstruct input
        reconstructed = self._decoder(latent.view(self._latent_array_dims))

        return reconstructed

In [45]:
# Initialize model and move to device
# ae_model = Era5AutoEncoder(wb_train_ds.num_channels, True).to(device)
imerg_autoencoder = ImergAutoEncoder(1, False).to(device)

It is useful at this point to ensure that we have correctly structured and dimensioned our layers by doing a forward pass on our data. If there is a mismatch between layers, we will find this error before we try to start training.

In [46]:
imerg_autoencoder._encoder(sample_tensor).view((-1,32,16,16))

tensor([[[[0.0000, 0.0311, 0.0333,  ..., 0.0333, 0.0333, 0.0333],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.0355, 0.0163, 0.0159,  ..., 0.0159, 0.0159, 0.0159],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0

In [47]:
sample_tensor.shape,imerg_autoencoder.forward(sample_tensor).shape

(torch.Size([1, 1, 64, 64]), torch.Size([1, 1, 64, 64]))

### Set up and run the training loop
Now we set up the loop to run gradient descent with back propogation to optimise the model weights of the autoencoder

In [48]:
# Loss function and optimizer
# loss_function = torch.nn.L1Loss()
# criterion = nn.KLDivLoss()
loss_function = torch.nn.MSELoss()

optimizer = torch.optim.Adam(imerg_autoencoder.parameters(), 
                             lr=5e-3)

In [49]:
num_epochs = 25

In [50]:
num_samples = len(ew4_imerg_train_pipe)
batch_size=8
num_batches = math.ceil(num_samples / batch_size)
num_batches

183

In [51]:
def get_batch_tensors(batch_size, ds_iterator, device):
    """
    Get a torch tensor for target and predictors from a pyearthtools pipeline iterator
    """
    predictor_batch_list = []
    target_batch_list = []
    for _ in range(batch_size):
        pred_sample, target_sample = next(ds_iterator)
        predictor_batch_list += [pred_sample[0]]
        target_batch_list += [target_sample[0]]
    
    # create numpy array of a batch 
    predictor_array = numpy.concat(predictor_batch_list, axis=0)
    target_array = numpy.concat(target_batch_list, axis=0)
        
    # convert to a tensor and send to gpu
    predictor_gpu_tensor = torch.tensor(
        predictor_array,
        dtype=torch.float32,
    ).to(device)
    target_gpu_tensor = torch.tensor(
        target_array,
        dtype=torch.float32,
    ).to(device)    
    
    return predictor_gpu_tensor, target_gpu_tensor
        
    

To be sure of our model set up, run a forward pass with a batch to check it works as it did with a single sample

In [52]:
myit = iter(ew4_imerg_train_pipe)
imerg_autoencoder.forward(get_batch_tensors(batch_size, myit, device)[0]).shape, imerg_autoencoder.forward(get_batch_tensors(batch_size, myit, device)[0]).shape

(torch.Size([8, 1, 64, 64]), torch.Size([8, 1, 64, 64]))

In [ ]:
%%time
for epoch_num in range(num_epochs):
    print(epoch_num)
    epoch_train_loss = 0.0
    epoch_val_loss = 0.0
    
    imerg_train_iter = iter(ew4_imerg_train_pipe)
    
    for batch_ix in range(num_batches):

        predictor_gpu_tensor, target_gpu_tensor = get_batch_tensors(batch_size, imerg_train_iter, device)
        
        if (batch_ix % 100) == 0:
            print(batch_ix)

        # do training for batch
        optimizer.zero_grad()
        predictions = imerg_autoencoder.forward(predictor_gpu_tensor)
        loss_batch = loss_function(predictions, target_gpu_tensor)
        loss_batch.backward()
        optimizer.step()
        epoch_train_loss += loss_batch.to('cpu').item()
    epoch_train_loss /= num_batches

    # calculate loss on validation data    
    for val_predictor, val_target in ew4_imerg_val_pipe:
        val_pred_tensor = torch.tensor(val_predictor[0], dtype=torch.float32).to(device)
        predictions_val = imerg_autoencoder.forward( val_pred_tensor)
        val_target_tensor = torch.tensor(val_target[0], dtype=torch.float32).to(device)
        loss_batch_val = loss_function(predictions_val, val_target_tensor)
        epoch_val_loss += loss_batch_val.to('cpu').item()
    
    epoch_val_loss /= len(ew4_imerg_val_pipe)
    
    print(epoch_train_loss)
    print(epoch_val_loss)
        
        
        
        


0
0
100
0.18664211530971234
0.035534426328736754
1
0
100
0.049715549606178466
0.020554312675265328
2
0
100
0.037770996821911935
0.015447328697939489
3
0


## Evaluate results using Scores

In [ ]:
import scores

In [ ]:
val_rmse_list = []
for predictor, target in ew4_imerg_val_pipe:
    val_prediction_arr = imerg_autoencoder.forward(torch.tensor(predictor[0], dtype=torch.float32).to(device)).to("cpu").detach().numpy()
    val_rmse_list += [scores.continuous.rmse(predictor[0],
                                             val_prediction_arr)]

In [ ]:
val_rmse = numpy.mean(val_rmse_list)

In [ ]:
val_rmse

In [ ]:
ew4_imerg_pipeline['2025-07-23 15:00'][0][0]['precipitation']

In [ ]:
 predictor, target = ew4_imerg_val_pipe['2025-07-23 15:00']

In [ ]:
pred_gpu = target_gpu = torch.tensor(
    predictor[0],
    dtype=torch.float32,
).to(device)
target_gpu = torch.tensor(
    target[0],
    dtype=torch.float32,
).to(device)

In [ ]:
model_prediction_array = imerg_autoencoder.forward(torch.tensor(
    predictor[0],
    dtype=torch.float32,
).to(device)).to("cpu").detach().numpy()

Our model prediction will be output as a numpy array (after we've extracted it from pytorch tensor). We want to then reinstate the metadata so we can interact with the model predictions, for example showing plots. We can use the pipeline reverse functionality to reverse the to numpy operation and get an xarray dataset out.

In [ ]:
input_ds = ew4_imerg_ml.reversed(predictor[0])
truth_ds = ew4_imerg_ml.reversed(target[0])
model_prediction = ew4_imerg_ml.reversed(model_prediction_array)

In [ ]:
(model_prediction['precipitation'] - truth_ds['precipitation']).plot.hist()

In [ ]:
plot_kwargs = {'cmap':'viridis', 'vmin':0.0, 'vmax':5.0}

In [ ]:
(model_prediction['precipitation'] - truth_ds['precipitation'])[0].plot.contourf(**plot_kwargs)

In [ ]:
fig1 = matplotlib.pyplot.figure(figsize=(18,6))

ax1=fig1.add_subplot(1,3,1,projection=cartopy.crs.PlateCarree())
input_ds['precipitation'][0].plot.contourf(ax=ax1, **plot_kwargs)
ax1.coastlines(color='w')

ax1=fig1.add_subplot(1,3,2,projection=cartopy.crs.PlateCarree())
model_prediction['precipitation'][0].plot.contourf(ax=ax1, **plot_kwargs)
ax1.coastlines(color='w')


ax1=fig1.add_subplot(1,3,3,projection=cartopy.crs.PlateCarree())
truth_ds['precipitation'][0].plot.contourf(ax=ax1, **plot_kwargs)
ax1.coastlines(color='w')


## Further Links

* [PyEarthTools Docs](https://pyearthtools.readthedocs.io/en/latest/)
  * [Tutorial Gallery](https://pyearthtools.readthedocs.io/en/latest/notebooks/Gallery.html)
* [PyEarthTools Repo](https://github.com/ACCESS-Community-Hub/PyEarthTools)
* [PyEarthTools JASMIN Site Archive Repo](https://github.com/MetOffice/pyearthtools_jasmin/)
*  [IMERG Dataset Info](https://gpm.nasa.gov/data/imerg)